# Imports

In [ ]:
# For development: enable auto-reload of modules
%load_ext autoreload
%autoreload 2

In [ ]:
import functools

import nibabel as nb
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from matplotlib import cm
from matplotlib import pyplot as plt
from nilearn import plotting as nlp
from scipy import stats
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)
cp = sns.color_palette()
%matplotlib inline

# Set up

## Set notebook parameters

In [ ]:
# Import configuration and utilities from package
import dl_morphometrics_helpers.constants as cfg
import dl_morphometrics_helpers.data_processing as du

## Load global volumes

In [ ]:
# Use helper function instead of manual loading and merging
merged = du.load_and_merge_brain_confounds()

## Load session and age info

In [ ]:
ses_info = pd.read_csv(cfg.age_tsv, sep="\t")

In [ ]:
# Use helper function instead of manual parsing
ses_info = du.parse_bids_ids(ses_info)
ses_info["age_years"] = ses_info.age / 12
pml = len(merged)
with_age = merged.merge(
    ses_info.loc[:, cfg.metadata_cols], how="left", on=["subject", "session"]
)
assert pml == len(with_age)

In [ ]:
sex_df = ses_info.query("sex.notnull()").groupby("subject").sex.first().reset_index()
with_sex = with_age.merge(sex_df, how="left", on="subject")

## get ages for last wave

In [ ]:
fastages = pd.read_csv(cfg.fastages_tsv, sep="\t")
# Use helper function instead of manual parsing
fastages = du.parse_bids_ids(fastages)
fastages["age_years"] = fastages.age / 12

with_all_ages = with_sex.merge(
    fastages.loc[:, cfg.metadata_cols],
    how="left",
    on=["subject", "session"],
    suffixes=["", "_fa"],
)
with_all_ages["age_years"] = with_all_ages.age_years.fillna(with_all_ages.age_years_fa)
assert len(with_all_ages.loc[with_all_ages.age_years.isnull()]) == 0

## Global-percent-volume differences

### Compute differences

In [ ]:
# compute all pct‐diff columns into a dict of Series
pct_diff_cols = {}
for tag, (base_suf, comp_suf) in cfg.cfg.diff_specs.items():
    for mc in cfg.cfg.metric_cols_nospace:
        base_col = f"{mc}{base_suf}"
        comp_col = f"{mc}{comp_suf}"
        pct_col = f"{mc}_pct{base_suf}_dif{comp_suf}"
        pct_diff_cols[pct_col] = (
            (with_all_ages[comp_col] - with_all_ages[base_col])
            / with_all_ages[base_col]
        ) * 100


# 3) make a single DataFrame and concat once
new_cols = pd.DataFrame({**pct_diff_cols}, index=with_all_ages.index)
global_df = pd.concat([with_all_ages, new_cols], axis=1)

global_df

### Compute models

In [ ]:
df = global_df.drop(columns="site_id sex age_years_fa site_id_fa".split()).copy()
nans_found = df.isna().sum().loc[lambda s: s > 0]
assert not len(nans_found)

# remove some stray cols with zero variance... only required with ranot2
# zero_var = df.iloc[:,2:].var(axis=0) == 0
# zvar_cols = zero_var[zero_var].index.tolist()
# assert len(zvar_cols) < 5
# df = df.drop(columns=zvar_cols)
# print(f"Zero var cols: {zvar_cols}")

# 3) center age at 9
df["age_centered"] = df["age_years"] - 9


# 4) fit & collect
all_results = []
model_summaries = {}
for scenario, (base_suf, comp_suf) in cfg.cfg.diff_specs.items():
    diff_suffix = f"_pct{base_suf}_dif{comp_suf}"
    for metric in cfg.cfg.metric_cols_nospace:
        colname = f"{metric}{diff_suffix}"
        if colname not in df:
            print(f"Missing {colname}")
            continue
        # formula: <metric>_pct_<pipeline1>_dif_<pipeline2> ~ age_years
        formula = f"{colname} ~ age_years"
        model = smf.ols(formula, data=df).fit()
        tbl = (
            model.summary2()
            .tables[1]
            .reset_index()
            .rename(columns={"index": "variable"})
        )
        tbl["metric"] = metric
        tbl["scenario"] = scenario
        tbl["scenario_label"] = cfg.cfg.comparison_labels.get(scenario, scenario)
        all_results.append(tbl)
        model_summaries[colname] = model.summary()

global_resses = pd.concat(all_results, ignore_index=True)
global_resses

In [ ]:
model_summaries.keys()

## Regionwise volume differences

### Compute diffs and residuals

In [ ]:
# -----------------------------------------------------------------------------
# Some settings -------------------------------------------------------------
# -----------------------------------------------------------------------------
atlas = "Desikan2006"


# -----------------------------------------------------------------------------
# HELPERS ---------------------------------------------------------------------
# -----------------------------------------------------------------------------
def read_grayvol(pipeline: str, suffix: str) -> pd.DataFrame:
    tsv = cfg.stats_dir / pipeline / f"atlas-{atlas}_GrayVol.tsv"
    df = (
        pd.read_csv(tsv, sep="\t")
        # .drop("temporalpole_rh", axis=1, errors="ignore")
    )

    df.columns = ["subject", "session", *[c + suffix for c in df.columns[2:]]]

    return df


def build_grayvol_df() -> pd.DataFrame:
    merged = read_grayvol("recon-all", "_ra")

    for p, suf, how in cfg.cfg.pipelines:
        if suf == "_ra":
            continue
        merged = merged.merge(read_grayvol(p, suf), on=["subject", "session"], how=how)
    return merged


def add_pct_diffs(df: pd.DataFrame) -> pd.DataFrame:
    """Compute percentage-difference columns for *every* metric, batch-joined once."""
    new_cols = {}
    for _key, (base_suf, comp_suf) in cfg.cfg.diff_specs.items():
        base_cols = [
            c for c in df.columns if c.endswith(base_suf) and c not in cfg.metadata_cols
        ]
        for base_col in base_cols:
            metric = base_col[: -len(base_suf)]
            comp_col = f"{metric}{comp_suf}"
            if comp_col in df.columns:
                pct_name = f"{metric}_pct{base_suf}_dif{comp_suf}"
                new_cols[pct_name] = (
                    (df[comp_col] - df[base_col]) / df[base_col]
                ) * 100
            else:
                raise ValueError(f"oops... {comp_col}")

    if new_cols:
        # build a DataFrame of all new columns at once
        pct_df = pd.DataFrame(new_cols, index=df.index)
        # one-shot concat avoids fragmentation
        df = pd.concat([df, pct_df], axis=1)
    return df


def run_regression(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for key, (base_suf, comp_suf) in cfg.cfg.diff_specs.items():
        tag = f"_pct{base_suf}_dif{comp_suf}"
        for y in (c for c in df.columns if c.endswith(tag)):
            res = smf.ols(f"Q('{y}') ~ age_years", data=df).fit()
            summ = (
                res.summary2()
                .tables[1]
                .reset_index()
                .rename(columns={"index": "term"})
                .assign(metric=y.replace(tag, ""), dif=key)
            )
            rows.append(summ)
    if not rows:
        raise ValueError("No regression models were fitted.")
    return pd.concat(rows, ignore_index=True)


# -----------------------------------------------------------------------------
# PIPELINE --------------------------------------------------------------------
# -----------------------------------------------------------------------------
_gv = build_grayvol_df().merge(
    with_all_ages[cfg.metadata_cols], on=["subject", "session"], how="left"
)

surfaces_df = add_pct_diffs(_gv)

rgn_resses = run_regression(surfaces_df)

# 3. FDR correction (batch the assignments)
# prepare full-length arrays
q_all = np.full(len(rgn_resses), np.nan, dtype=float)
sig_all = np.full(len(rgn_resses), False, dtype=bool)

for term in ["age_years", "Intercept"]:
    mask = rgn_resses["term"] == term
    sig, q, *_ = multipletests(rgn_resses.loc[mask, "P>|t|"], method="fdr_by")
    sig_all[mask] = sig
    q_all[mask] = q

# single shot assign
rgn_resses = rgn_resses.assign(q=sig_all, fdr_sig=sig_all)

In [ ]:
rgn_resses

### Create tr/te df with age correction

In [ ]:
fix_df = (
    surfaces_df.copy()
    .reset_index(drop=True)
    .assign(centered_age=lambda df: df.age_years - df.age_years.mean())
)
fix_df["centered_age_squared"] = fix_df.centered_age**2
fix_df["age_years_squared"] = fix_df.age_years**2

fix_df["centered_age_cubed"] = fix_df.centered_age**3
region_names = du.get_region_names(surfaces_df)

In [ ]:
tr_df = fix_df.sample(frac=0.5)
te_df = fix_df.loc[~fix_df.index.isin(tr_df.index)].copy()

tr_df.age_years.describe()

In [ ]:
fix_resses = []
z_df = fix_df.copy()

for pipeline, short, _ in cfg.cfg.pipelines:
    source = short.strip("_")
    tmp = []
    for mc in region_names:
        z_df[f"{mc}_{source}"] = (
            z_df[f"{mc}_{source}"] - z_df[f"{mc}_{source}"].mean()
        ) / z_df[f"{mc}_{source}"].std()
        # tr label cruft from elsewhere
        trmdl = smf.rlm(f"{mc}_{source} ~ age_years + age_years_squared", data=z_df)
        trmdl_fitted = trmdl.fit()
        trres = pd.DataFrame(trmdl_fitted.summary2().tables[1].to_dict())
        trres["source"] = source
        trres["region"] = mc
        trres["split"] = "full"
        trres["sig"] = trres["P>|z|"] < 0.05
        trres["sign"] = np.sign(trres["Coef."])
        trres = trres.reset_index(names=["term"])
        tmp.append(trres)
    tmp = pd.concat(tmp)
    sig, q, _, _ = multipletests(
        tmp.query('term == "Intercept"')["P>|z|"], method="fdr_by"
    )
    tmp.loc[tmp.term == "Intercept", "q"] = q
    tmp.loc[tmp.term == "Intercept", "fdr_sig"] = sig

    sig, q, _, _ = multipletests(
        tmp.query('term == "age_years"')["P>|z|"], method="fdr_by"
    )
    tmp.loc[tmp.term == "age_years", "q"] = q
    tmp.loc[tmp.term == "age_years", "fdr_sig"] = sig

    sig, q, _, _ = multipletests(
        tmp.query('term == "age_years_squared"')["P>|z|"], method="fdr_by"
    )
    tmp.loc[tmp.term == "age_years_squared", "q"] = q
    tmp.loc[tmp.term == "age_years_squared", "fdr_sig"] = sig
    fix_resses.append(tmp)
fix_resses = pd.concat(fix_resses)
fix_resses = fix_resses.rename(columns={"Coef.": "coef", "P>|z|": "p"})

# pvals
age_ps = fix_resses.loc[
    fix_resses.term == "age_years", ["source", "region", "p"]
].pivot(columns=["source"], index="region")
age_ps.columns = [cc[1] for cc in age_ps.columns]

age_coefs = fix_resses.loc[
    fix_resses.term == "age_years", ["source", "region", "coef"]
].pivot(columns=["source"], index="region")
age_coefs.columns = [cc[1] for cc in age_coefs.columns]

age2_coefs = fix_resses.loc[
    fix_resses.term == "age_years_squared", ["source", "region", "coef"]
].pivot(columns=["source"], index="region")
age2_coefs.columns = [cc[1] for cc in age2_coefs.columns]

# Figures

## Create stats table (Figure 1C)


In [ ]:
# -----------------------------------------------------------------------------
# BUILD TIDY REGRESSION TABLE --------------------------------------------------
# -----------------------------------------------------------------------------

label = "Comparison"

# Base tidy table with formatted stats (re‑usable for any subset)
base_tab = (
    global_resses.rename(
        columns={
            "variable": "Term",
            "Coef.": "Coef",
            "P>|t|": "p",
            "scenario_label": label,
        }
    )
    .assign(Measure=lambda df: df["metric"].map(cfg.cfg.metric_contractions))
    .dropna(subset=["Measure"])[["Measure", label, "Term", "Coef", "t", "p"]]
    .assign(
        Coef=lambda df: df["Coef"].map(lambda x: f"{x:.2f}"),
        t=lambda df: df["t"].map(lambda x: f"{x:.2f}"),
        p=lambda df: df["p"].map(lambda x: f"{x:.2g}*" if x < 0.05 else f"{x:.2g}"),
    )
)

measures = list(cfg.cfg.metric_contractions.values())
terms = ["Intercept", "age_years"]
row_order = pd.MultiIndex.from_product([measures, terms], names=["Measure", "Term"])
stats_order = ["Coef", "t", "p"]


def make_stats_df(group_keys):
    """Return a tidy *wide* DataFrame (nested indices) for the given cfg.pipelines."""
    scenarios = [cfg.cfg.comparison_labels[k] for k in group_keys]

    tab = base_tab[base_tab[label].isin(scenarios)].copy()

    wide = tab.pivot_table(
        index=["Measure", "Term"],
        columns=label,
        values=["Coef", "t", "p"],
        aggfunc="first",
    ).swaplevel(0, 1, axis=1)
    # order rows / cols for consistent display
    col_order = pd.MultiIndex.from_product(
        [scenarios, stats_order], names=[label, "Stat"]
    )
    wide = wide.reindex(index=row_order, columns=col_order)
    return wide


def df_to_table_fig(df: pd.DataFrame, title: str):
    """Convert nested‑index DataFrame to a matplotlib table figure."""
    n_rows, n_cols = df.shape
    fig, ax = plt.subplots(figsize=(1.1 + 0.9 * n_cols, 0.4 + 0.43 * n_rows), dpi=300)
    ax.axis("off")
    tbl = ax.table(
        cellText=df.reset_index().values,
        colLabels=list(df.reset_index().columns),
        loc="center",
        cellLoc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(6)
    tbl.scale(1, 1.2)
    fig.tight_layout()
    fig.suptitle(title, y=1.02, fontsize=10)
    return fig


# # Regression summary tables
df_tab_rac = make_stats_df(cfg.cfg.rac_groups)
df_tab_rany = make_stats_df(cfg.cfg.rany_groups)

fig_tab_rac = df_to_table_fig(df_tab_rac, "Regression summary – Recon‑all clinical")
fig_tab_rany = df_to_table_fig(df_tab_rany, "Regression summary – Recon‑any")

for fig, name in [(fig_tab_rac, "RAC_stats"), (fig_tab_rany, "RANY_stats")]:
    fig.savefig(
        f"../output/images/regression_table_{name}.png", dpi=600, bbox_inches="tight"
    )
    fig.savefig(f"../output/images/regression_table_{name}.pdf", bbox_inches="tight")

In [ ]:
df_tab_rany

In [ ]:
df_tab_rac

## Figure 1 A/B

In [ ]:
# -----------------------------------------------------------------------------
# SETUP --------------------------------------------------------------------
# -----------------------------------------------------------------------------

# Age colour‑map
age_norm = plt.Normalize(9, 18)
age_cmap = plt.get_cmap("flare")


# -----------------------------------------------------------------------------
# HELPER ----------------------------------------------------------------------
# -----------------------------------------------------------------------------


def plot_correlation_summary(groups, tag):
    """Scatter + %Δ figure for a list of pipeline *keys* (rows) and the global_df.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    rows, cols = (
        len(groups),
        len(cfg.cfg.metric_contractions) + 1,
    )  # +1 for summary panel
    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(4 * cols, 2.4 * rows),
        dpi=110,
        layout="constrained",
        sharex="col",
    )

    for r, g in enumerate(groups):
        g_lbl = cfg.comparison_labels[g]
        comp_suffix = cfg.diff_specs[g][1]

        # ---------------------------------------------------- scatter sub‑plots
        for c, m in enumerate(cfg.metric_contractions.keys()):
            ax = axes[r, c]
            x = global_df[f"{m}_ra"].to_numpy()
            y = global_df[f"{m}{comp_suffix}"].to_numpy()
            ok = ~(np.isnan(x) | np.isnan(y))
            r2 = np.corrcoef(x[ok], y[ok])[0, 1] ** 2 if ok.sum() else np.nan

            ax.scatter(
                x,
                y,
                c=global_df["age_years"],
                cmap=age_cmap,
                norm=age_norm,
                marker=".",
                s=9,
                rasterized=True,
            )
            lim = np.nanmax([x, y])
            ax.plot([0, lim], [0, lim], color="black", lw=0.8)

            if r == rows - 1:
                ax.set_xlabel("RA ($10^5$ mm$^3$)")
            if c == 0:
                ax.set_ylabel(f"{g_lbl} ($10^5$ mm$^3$)")
            ax.set_title(f"{cfg.metric_contractions[m]} • $R^2$={r2:.2f}", fontsize=8)

        # ---------------------------------------------------- %Δ summary panel
        ax_sum = axes[r, -1]
        for m in cfg.metric_contractions.keys():
            base = global_df[f"{m}_ra"]
            comp = global_df[f"{m}{comp_suffix}"]
            pct = 100 * (comp - base) / base
            ax_sum.scatter(
                global_df["age_years"],
                pct,
                s=9,
                alpha=0.6,
                label=cfg.metric_contractions[m],
            )

        ax_sum.axhline(0, color="black", lw=0.8)
        if r == rows - 1:
            ax_sum.set_xlabel("Age (years)")
        if r == 0:
            ax_sum.set_title("% Δ vs RA", fontsize=8)
            ax_sum.legend(fontsize=6, loc="upper right", markerscale=2)
        ax_sum.set_ylabel("")

    # Shared colour‑bar (age)
    fig.colorbar(
        cm.ScalarMappable(norm=age_norm, cmap=age_cmap),
        ax=axes[:, : len(cfg.metric_contractions)],
        location="bottom",
        shrink=0.6,
        label="Age (years)",
    )
    fig.suptitle(f"Comparison between {tag} and recon-all", y=1.02, fontsize=12)
    return fig


# -----------------------------------------------------------------------------
# BUILD & SAVE FIGURES ---------------------------------------------------------
# -----------------------------------------------------------------------------

fig_rac = plot_correlation_summary(cfg.rac_groups, "recon-all-clinical")
fig_rany = plot_correlation_summary(cfg.rany_groups, "recon-any")

for fig, name in [(fig_rac, "RAC"), (fig_rany, "RANY")]:
    fig.savefig(
        f"../output/images/correlation_summaries_{name}.png",
        dpi=600,
        bbox_inches="tight",
    )
    fig.savefig(
        f"../output/images/correlation_summaries_{name}.pdf", bbox_inches="tight"
    )

# Assess region wise gray volume (Figure 1D)

## surface plots 

In [ ]:
rgn_resses

In [ ]:
@functools.cache
def _load_surface(hemi: str):
    """Return (surf_mesh, sulc_map, label_data, lut_df) for a hemisphere.

    Loads files **directly from the local TemplateFlow cache** (`~/.cache/templateflow/tpl-fsaverage`).
    Ensure these files exist beforehand (e.g., via `templateflow fetch fsaverage`).
    """
    hemi_code = hemi.upper()[0]  # 'L' or 'R'

    mesh = cfg.tpl_root / f"tpl-fsaverage_hemi-{hemi_code}_164k_inflated.surf.gii"
    sulc = cfg.tpl_root / f"tpl-fsaverage_hemi-{hemi_code}_den-164k_sulc.shape.gii"
    lab = (
        cfg.tpl_root
        / f"tpl-fsaverage_hemi-{hemi_code}_den-164k_atlas-{atlas}_seg-aparc_desc-curated_dseg.label.gii"
    )
    lut = cfg.tpl_root / f"tpl-fsaverage_atlas-{atlas}_dseg.tsv"

    if not lab.exists():
        raise FileNotFoundError(
            f"{lab} not found. Populate your cache or adjust the density/atlas."
        )

    mesh = str(mesh)
    sulc = str(sulc)
    lab_img = nb.load(str(lab))

    lut_df = (
        pd.read_csv(lut, sep="	")
        .query("hemi == @hemi_code")
        .reset_index(names="val")
        .assign(name_h=lambda d: d.name.str.lower() + f"_{hemi.lower()}")
    )
    lut_df_raw = pd.read_csv(lut, sep="\t").query("hemi == @hemi_code")
    if hemi_code == "R":
        lut_df = lut_df_raw.reset_index(drop=True).reset_index(names="val")
    else:
        lut_df = lut_df_raw.reset_index(names="val")
    lut_df["name_h"] = lut_df.name.str.lower() + f"_{hemi.lower()}"

    return mesh, sulc, lab_img.agg_data(), lut_df


def surf_coeff_array(resses: pd.DataFrame, diff_key: str, hemi: str) -> np.ndarray:
    """Map coefficients for age_years term onto fsaverage mesh indices."""
    mesh, sulc, lab_data, lut = _load_surface(hemi)
    df = resses.query("term == 'age_years' & dif == @diff_key").merge(
        lut[["val", "name_h"]], how="left", left_on="metric", right_on="name_h"
    )
    arr = np.zeros_like(lab_data, dtype=float)
    for _, row in df.iterrows():
        arr[lab_data == row.val] = float(row["Coef."])
    return arr


def plot_surface(resses, diff_key: str, vmax: float = 1.0):
    """4‑view surface figure for a given diff key."""
    lh_arr = surf_coeff_array(resses, diff_key, "lh")
    rh_arr = surf_coeff_array(resses, diff_key, "rh")
    fig, axes = plt.subplots(
        1, 4, subplot_kw={"projection": "3d"}, figsize=(10, 5), dpi=120
    )
    cfgs = [
        ("lh", lh_arr, "lateral", axes[0]),
        ("lh", lh_arr, "medial", axes[1]),
        ("rh", rh_arr, "lateral", axes[2]),
        ("rh", rh_arr, "medial", axes[3]),
    ]
    for hemi, arr, view, ax in cfgs:
        mesh, sulc, _, _ = _load_surface(hemi)
        nlp.plot_surf_stat_map(
            mesh,
            arr,
            bg_map=sulc,
            bg_on_data=True,
            cmap="RdBu_r",
            view=view,
            axes=ax,
            vmax=vmax,
            vmin=-vmax,
            colorbar=(ax is axes[-1]),
        )
    fig.suptitle(f"Surface age β – {cfg.comparison_labels[diff_key]}")
    return fig


for key in "ct1 anyt1".split():
    fig = plot_surface(rgn_resses, key, vmax=1)
    tag = cfg.comparison_labels[key]
    fig.savefig(
        f"../output/images/surface_age_beta_{tag}.png", dpi=600, bbox_inches="tight"
    )
    fig.savefig(f"../output/images/surface_age_beta_{tag}.pdf", bbox_inches="tight")

# How badly does the error mess up models of development

## Figure 2

## OLS estimation Global-Volume/Age (2A)

In [ ]:
def plot_global_volume_age_modeling(suffixes, tag):
    ncols = len(cfg.metric_contractions)
    fig, axes = plt.subplots(
        1,
        ncols,
        figsize=(4 * ncols, 2.4 * 1),
        dpi=110,
        layout="constrained",
        sharex="col",
    )
    for col_idx, metric in enumerate(cfg.metric_contractions.keys()):
        ax = axes[col_idx]
        for _line_idx, suffix in enumerate(suffixes):
            pipeline = suffix.strip()

            sns.regplot(
                data=global_df,
                x="age_years",
                y=f"{metric}{suffix}",
                scatter=False,
                ax=ax,
                label=pipeline,
            )

        ax.set_ylabel(f"{cfg.metric_contractions[metric]} ($10^5$ mm$^3$)")
        ax.set_xlabel("Age (Years)")
        # ax.set_title(f"{cfg.metric_contractions[m]} • $R^2$={r2:.2f}", fontsize=8)

    fig.suptitle(f"Volume age comparisons for {tag}", y=1.02, fontsize=12)
    fig.legend(
        [s.strip("_") for s in suffixes],
        loc="lower center",
        bbox_to_anchor=[0.5, 0.53],
        ncols=4,
    )

    return fig


rac_suffixes = ["_ra", *[x[1] for x in cfg.pipelines[1:5]]]
rany_suffixes = ["_ra", *[x[1] for x in cfg.pipelines[5:]]]
fig = plot_global_volume_age_modeling(rac_suffixes, "recon-all clinical")

In [ ]:
fig = plot_global_volume_age_modeling(rany_suffixes, "recon-any")

## Robust regression Region-wise/Age (2B)

In [ ]:
def plot_regionwise_volume_age_modeling(groups, tag):
    ncols = len(groups)
    fig, axes = plt.subplots(
        1,
        ncols,
        figsize=(4 * ncols, 2.4 * 1),
        dpi=110,
        layout="constrained",
        sharex="col",
    )
    diffs_for_plotting = {k: v for k, v in cfg.diff_specs.items() if k in groups}
    for col_idx, (key, diff) in enumerate(diffs_for_plotting.items()):
        # print(diff)
        ax = axes[col_idx]
        base = diff[0].strip("_")
        comp = diff[1].strip("_")
        sns.regplot(
            data=age_coefs,
            x=base,
            y=comp,
            robust=True,
            ax=ax,
            color=cp[col_idx + 1],
            label=cfg.comparison_labels[key],
        )
        ax.get_xlim()
        ax.get_ylim()
        ax.plot([-1, 1], [-1, 1], color="black")
        # ax.set_xlim(xlim)
        # ax.set_ylim(ylim)
        ax.set_xlabel(r"$\beta_{age}$ from " + base.upper())
        ax.set_ylabel(r"$\beta_{age}$ from " + comp.upper())
        r2 = np.corrcoef(age_coefs[base], age_coefs[comp])[0, 1]
        ax.set_title(f"$r^2$ = {r2:0.2f}")

    fig.suptitle(f"Region-wise-volume-age comparisons for {tag}", y=1.02, fontsize=12)
    # return fig


plot_regionwise_volume_age_modeling(cfg.rac_groups, "recon-all clinical")

In [ ]:
plot_regionwise_volume_age_modeling(cfg.rany_groups, "recon-any")

In [ ]:
stats.pearsonr(
    fix_resses.query('term == "age_years" & source=="ra"').coef,
    fix_resses.query('term == "age_years" & source=="ract1r5"').coef,
)

In [ ]:
stats.pearsonr(
    fix_resses.query('term == "age_years" & source=="ra"').coef,
    fix_resses.query('term == "age_years" & source=="ract2"').coef,
)

# Modeling the effect of site

In [ ]:
siteresses = []
for tag, (base_suf, comp_suf) in cfg.diff_specs.items():
    for mc in region_names:
        base_col = f"{mc}{base_suf}"
        comp_col = f"{mc}{comp_suf}"
        dif = f"{base_suf}_dif{comp_suf}"
        pct_col = f"{mc}_pct{dif}"
        mdl = smf.ols(f"{pct_col} ~ centered_age*site_id", data=fix_df)

        mdl_fitted = mdl.fit()
        mdl_fitted.summary()
        res = (
            mdl_fitted.summary2()
            .tables[1]
            .reset_index()
            .rename(columns={"index": "vn"})
        )
        res["metric"] = mc
        res["dif"] = dif
        siteresses.append(res)
siteresses = pd.concat(siteresses)

sig, q, _, _ = multipletests(
    siteresses.query('vn == "centered_age"')["P>|t|"], method="fdr_by"
)
siteresses.loc[siteresses.vn == "centered_age", "q"] = q
siteresses.loc[siteresses.vn == "centered_age", "fdr_sig"] = sig

sig, q, _, _ = multipletests(
    siteresses.query('vn == "Intercept"')["P>|t|"], method="fdr_by"
)
siteresses.loc[siteresses.vn == "Intercept", "q"] = q
siteresses.loc[siteresses.vn == "Intercept", "fdr_sig"] = sig

sig, q, _, _ = multipletests(
    siteresses.loc[siteresses.vn.str.contains("site"), "P>|t|"], method="fdr_by"
)
siteresses.loc[siteresses.vn.str.contains("site"), "q"] = q
siteresses.loc[siteresses.vn.str.contains("site"), "fdr_sig"] = sig

# site effects mostly don't surive multiple comparisons

In [ ]:
siteresses.loc[siteresses.vn.str.contains("site") & siteresses.fdr_sig, :].sort_values(
    "vn"
)

# TR and TE res

I'm not sure what's happening here. It looks like it is an exploration of how modeling is impacted...

Not used for any of the figures for the poster abstract

### Compute tr/te resses

In [ ]:
raise AssertionError()
tr_resses = []
te_resses = []
sources = ["ra", "ract1", "ract1r3", "ract1r5", "ract2"]
for source in sources:
    for mc in cfg.metric_cols:
        trmdl = smf.ols(f"{mc}_{source} ~ age_years + age_years_squared", data=tr_df)
        trmdl_fitted = trmdl.fit()
        temdl = smf.ols(f"{mc}_{source} ~ age_years + age_years_squared", data=te_df)
        temdl_fitted = temdl.fit()
        trres = pd.DataFrame(trmdl_fitted.summary2().tables[1].to_dict())
        trres["source"] = source
        trres["region"] = mc
        trres["split"] = "train"
        trres["sig"] = trres["P>|t|"] < 0.05
        trres["sign"] = np.sign(trres["Coef."])
        trres = trres.reset_index(names=["term"])
        tr_resses.append(trres)
        teres = pd.DataFrame(temdl_fitted.summary2().tables[1].to_dict())
        teres["source"] = source
        teres["region"] = mc
        teres["split"] = "train"
        teres["sig"] = teres["P>|t|"] < 0.05
        teres["sign"] = np.sign(teres["Coef."])
        teres = teres.reset_index(names=["term"])
        te_resses.append(teres)
tr_resses = pd.concat(tr_resses)
te_resses = pd.concat(te_resses)

In [ ]:
tr_resses.query("sig & term != 'Intercept'")

In [ ]:
te_resses.query("sig & term != 'Intercept'")

### Model residuals

In [ ]:
mdl_resses = []
sources = ["ract1", "ract1r3", "ract1r5", "ract2"]
for train_source in sources:
    for test_source in sources:
        if test_source != train_source:
            continue
        tmptedf = te_df.copy()
        for mc in cfg.metric_cols:
            mdl = smf.ols(
                f"{mc}_ra ~ centered_age*{mc}_{train_source} + centered_age_squared*{mc}_{train_source}",
                data=tr_df,
            )
            mdl_fitted = mdl.fit()
            tmptedf[f"{mc}_{train_source}"] = tmptedf[f"{mc}_{test_source}"]
            pred = mdl_fitted.predict(tmptedf)
            error = pred - te_df[f"{mc}_ra"]
            pct_error = (error / te_df[f"{mc}_ra"]) * 100
            nomdl_error = tmptedf[f"{mc}_{train_source}"] - te_df[f"{mc}_ra"]
            pct_nomdl_error = (nomdl_error / te_df[f"{mc}_ra"]) * 100
            null_error = tr_df[f"{mc}_ra"].mean() - te_df[f"{mc}_ra"]
            pct_null_error = (null_error / te_df[f"{mc}_ra"]) * 100
            tmpdf = te_df.loc[:, ["centered_age"]].copy()
            tmpdf["error"] = error
            emdl = smf.ols("error ~ centered_age", data=tmpdf)
            emdl_fitted = emdl.fit()
            res_age_terms = (
                emdl_fitted.summary2().tables[1].loc["centered_age"].to_dict()
            )
            tmpdf["ra"] = tmptedf[f"{mc}_ra"]
            tmpdf["pred"] = pred
            tmpdf["orig"] = te_df[f"{mc}_{test_source}"]
            mdlra = smf.ols("ra ~ centered_age", data=tmpdf)
            mdlra_fitted = mdlra.fit()
            mdlprd = smf.ols("pred ~ centered_age", data=tmpdf)
            mdlprd_fitted = mdlprd.fit()
            mdlorg = smf.ols("orig ~ centered_age", data=tmpdf)
            mdlorg_fitted = mdlorg.fit()

            mdl_res = {
                "target": "ra",
                "train_source": train_source,
                "test_source": test_source,
                "metric": mc,
                "mae_mdl": np.abs(error).mean(),
                "mae_nomdl": np.abs(nomdl_error).mean(),
                "mae_null": np.abs(null_error).mean(),
                "mape_mdl": np.abs(pct_error).mean(),
                "mape_nomdl": np.abs(pct_nomdl_error).mean(),
                "mape_null": np.abs(pct_null_error).mean(),
                "r2_mdl": 1 - ((error**2).sum() / ((null_error**2).sum())),
                "r2_nomdl": 1 - ((nomdl_error**2).sum() / ((null_error**2).sum())),
                "r2_mdl_v_nodml": 1 - ((error**2).sum() / ((nomdl_error**2).sum())),
                "res_age_r2": emdl_fitted.rsquared,
                "ra_age_b": mdlra_fitted.params["centered_age"],
                "ra_age_p": mdlra_fitted.pvalues["centered_age"],
                "prd_age_b": mdlprd_fitted.params["centered_age"],
                "prd_age_p": mdlprd_fitted.pvalues["centered_age"],
                "org_age_b": mdlorg_fitted.params["centered_age"],
                "org_age_p": mdlorg_fitted.pvalues["centered_age"],
            }
            # mdl_res.update(res_age_terms)
            mdl_resses.append(mdl_res)
mdl_resses = pd.DataFrame(mdl_resses)

In [ ]:
alpha = 0.05
mdl_resses["pct_prd_vs_ra_b"] = (
    (mdl_resses.prd_age_b - mdl_resses.ra_age_b) / mdl_resses.ra_age_b
) * 100
mdl_resses["pct_org_vs_ra_b"] = (
    (mdl_resses.org_age_b - mdl_resses.ra_age_b) / mdl_resses.ra_age_b
) * 100
mdl_resses["prd_TP"] = (mdl_resses.ra_age_p < alpha) & (mdl_resses.prd_age_p < alpha)
mdl_resses["prd_TN"] = (mdl_resses.ra_age_p > alpha) & (mdl_resses.prd_age_p > alpha)
mdl_resses["prd_FP"] = (mdl_resses.ra_age_p > alpha) & (mdl_resses.prd_age_p < alpha)
mdl_resses["prd_FN"] = (mdl_resses.ra_age_p < alpha) & (mdl_resses.prd_age_p > alpha)
mdl_resses["org_TP"] = (mdl_resses.ra_age_p < alpha) & (mdl_resses.org_age_p < alpha)
mdl_resses["org_TN"] = (mdl_resses.ra_age_p > alpha) & (mdl_resses.org_age_p > alpha)
mdl_resses["org_FP"] = (mdl_resses.ra_age_p > alpha) & (mdl_resses.org_age_p < alpha)
mdl_resses["org_FN"] = (mdl_resses.ra_age_p < alpha) & (mdl_resses.org_age_p > alpha)

In [ ]:
mdl_resses["delta_mape"] = mdl_resses.mape_mdl - mdl_resses.mape_nomdl

In [ ]:
mdl_resses.query("metric == 'G_and_S_frontomargin_lh'")

In [ ]:
mdl_resses.query("metric == 'G_and_S_occipital_inf_lh'")

In [ ]:
mdl_resses.groupby(["train_source", "test_source"]).prd_TP.mean()

In [ ]:
mdl_resses.groupby(["train_source", "test_source"]).org_TP.mean()

In [ ]:
mdl_resses.query("train_source == test_source").groupby(
    "train_source"
).r2_mdl.describe()

In [ ]:
mdl_resses.query("train_source == test_source").groupby(
    "train_source"
).delta_mape.describe()

In [ ]:
mdl_resses.query('train_source == "ract1"').groupby("test_source").r2_mdl.describe()

In [ ]:
mdl_resses.query('train_source == "ract1"').groupby("test_source").r2_mdl.describe()

In [ ]:
r2_num = ((nomdl_error) ** 2).sum()
r2_den = (null_error**2).sum()
r2 = 1 - (r2_num / r2_den)
r2

In [ ]:
r2_num = ((error) ** 2).sum()
r2_den = (nomdl_error**2).sum()
r2 = 1 - (r2_num / r2_den)
r2

In [ ]:
plt.plot(te_df[f"{mc}_ra"], mdl_fitted.predict(te_df), "o")

In [ ]:
plt.plot(te_df[f"{mc}_ra"], te_df[f"{mc}_ract1"], "o")

In [ ]:
plt.plot(te_df.centered_age, te_df[f"{mc}_ra"] - mdl_fitted.predict(te_df), "o")

# Miscellaneous plots... volume by age etc.

## stray regionwise plots

In [ ]:
stats.pearsonr(
    fix_resses.query('term == "age_years" & source=="ra"').sign,
    fix_resses.query('term == "age_years" & source=="ract1"').sign,
)

In [ ]:
(
    fix_resses.query('term == "age_years" & source=="ra"').sign
    == fix_resses.query('term == "age_years" & source=="ract1"').sign
).mean()

In [ ]:
(
    fix_resses.query('term == "age_years" & source=="ra"').sign
    == fix_resses.query('term == "age_years" & source=="ract1r5"').sign
).mean()

In [ ]:
sns.regplot(
    x=fix_resses.query('term == "age_years" & source=="ra"').coef,
    y=fix_resses.query('term == "age_years" & source=="ract1"').coef,
)

In [ ]:
sns.regplot(z_df, x="age_years", y="rostralmiddlefrontal_rh_ra")

## to_corr, regionwise, ct1

In [ ]:
to_corr = rgn_resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1"').copy()
to_corr["region"] = to_corr.metric.str.split("_").str[0]
to_corr["hemisphere"] = to_corr.metric.str.split("_").str[1]

# todo: consider whether this should be in a different notebook
# to_corr = to_corr.merge(regional_peaks, how="left", on="region")

In [ ]:
to_corr.loc[to_corr.hemisphere == "lh", ["region", "hemisphere"]].merge(
    to_corr.loc[to_corr.hemisphere == "rh", ["region", "hemisphere"]],
    how="outer",
    on="region",
    indicator=True,
)

In [ ]:
stats.pearsonr(
    to_corr.query('hemisphere == "lh"')["Coef."],
    to_corr.query('hemisphere == "lh"')["volume_peak"],
)

In [ ]:
(-0.5852530548111721) ** 2

In [ ]:
stats.pearsonr(
    to_corr.query('hemisphere == "rh"')["Coef."],
    to_corr.query('hemisphere == "rh"')["volume_peak"],
)

In [ ]:
sns.lmplot(data=to_corr, x="volume_peak", y="Coef.", hue="hemisphere")

to_corr = rgn_resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1r5"').copy()
to_corr["region"] = to_corr.metric.str.split("_").str[0]
to_corr["hemisphere"] = to_corr.metric.str.split("_").str[1]

to_corr = to_corr.merge(regional_peaks, how="left", on="region")
print(
    stats.pearsonr(
        to_corr.query('hemisphere == "lh"')["Coef."],
        to_corr.query('hemisphere == "lh"')["volume_peak"],
    )
)
print(
    stats.pearsonr(
        to_corr.query('hemisphere == "rh"')["Coef."],
        to_corr.query('hemisphere == "rh"')["volume_peak"],
    )
)

sns.lmplot(data=to_corr, x="volume_peak", y="Coef.", hue="hemisphere")

to_corr.merge(regional_peaks, how="left", on="region")

rgn_resses.query('dif == "_pct_ra_dif_ct1" & vn == "Intercept"').sort_values("t")

rgn_resses.query('dif == "_pct_ra_dif_ct1" & vn == "centered_age"').sort_values(
    "t"
).to_feather(f"../data/derivatives/fs_stats/var-age_atlas-{atlas}.feather")

rgn_resses.query('dif == "_pct_ct1_dif_ct2" & vn == "centered_age"').sort_values("t")

## Single panels

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(
    global_df.age_years,
    global_df["Total cortical gray matter volume_pct_ra_dif_ract1"],
    ".",
    label="Cereb GMV CT1",
)
ax.plot(
    global_df.age_years,
    global_df["Total cortical gray matter volume_pct_ra_dif_ract1r5"],
    ".",
    label="Cort GMV CT1R5",
)

ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of rac from ra")
# ax.set_ylim((-20, 20))
fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(
    global_df.age_years,
    global_df["Subcortical gray matter volume_pct_ra_dif_ct1"],
    ".",
    label="Subcort GMV",
)
ax.plot(
    global_df.age_years,
    global_df["Total cortical gray matter volume_pct_ra_dif_ct1"],
    ".",
    label="Cort GMV",
)
ax.plot(
    global_df.age_years,
    global_df["Total cerebral white matter volume_pct_ra_dif_ct1"],
    ".",
    label="Cereb WMV",
)
ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of rac from ra")
# ax.set_ylim((-20, 20))
fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(
    global_df.age_years,
    global_df["Subcortical gray matter volume_pct_ra_dif_not2"],
    ".",
    label="Subcort GMV",
)
ax.plot(
    global_df.age_years,
    global_df["Total cortical gray matter volume_pct_ra_dif_not2"],
    ".",
    label="Cort GMV",
)
ax.plot(
    global_df.age_years,
    global_df["Total cerebral white matter volume_pct_ra_dif_not2"],
    ".",
    label="Cereb WMV",
)
ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of ranot2 from ra")
# ax.set_ylim((-100, 100))
fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(
    global_df.age_years,
    global_df["Subcortical gray matter volume_pct_ra_dif_ct2"],
    ".",
    label="Subcort GMV",
)
ax.plot(
    global_df.age_years,
    global_df["Total cortical gray matter volume_pct_ra_dif_ct2"],
    ".",
    label="Cort GMV",
)
ax.plot(
    global_df.age_years,
    global_df["Total cerebral white matter volume_pct_ra_dif_ct2"],
    ".",
    label="Cereb WMV",
)
ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of rac from ra")
# ax.set_ylim((-20, 20))
fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(
    global_df.age_years,
    global_df["Subcortical gray matter volume_ra"],
    ".",
    label="Subcort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_ra",
    scatter=False,
    ax=ax,
    color=cp[0],
)
ax.plot(
    global_df.age_years,
    global_df["Total cortical gray matter volume_ra"],
    ".",
    label="Cort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_ra",
    scatter=False,
    ax=ax,
    color=cp[1],
)

ax.plot(
    global_df.age_years,
    global_df["Total cerebral white matter volume_ra"],
    ".",
    label="Cereb WMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cerebral white matter volume_ra",
    scatter=False,
    ax=ax,
    color=cp[2],
)

ax.set_xlabel("Age (years)")
fig.legend()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5))
ax = axes[0, 0]
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_ra",
    scatter=False,
    ax=ax,
    label="RA",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_ract1",
    scatter=False,
    ax=ax,
    label="RACT1",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_ract1r5",
    scatter=False,
    ax=ax,
    label="RACT1-R5",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_ract2",
    scatter=False,
    ax=ax,
    label="RACT2",
)
ax.set_ylabel("Cortical GMV")
ax.set_xlabel("Age (Years)")

ax = axes[0, 1]
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cerebral white matter volume_ra",
    scatter=False,
    ax=ax,
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cerebral white matter volume_ract1",
    scatter=False,
    ax=ax,
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cerebral white matter volume_ract1r5",
    scatter=False,
    ax=ax,
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cerebral white matter volume_ract2",
    scatter=False,
    ax=ax,
)
ax.set_ylabel("Cerebral WMV")
ax.set_xlabel("Age (Years)")

ax = axes[0, 2]
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_ra",
    scatter=False,
    ax=ax,
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_ract1",
    scatter=False,
    ax=ax,
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_ract1r5",
    scatter=False,
    ax=ax,
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_ract2",
    scatter=False,
    ax=ax,
)
ax.set_ylabel("Subcortical GMV")
ax.set_xlabel("Age (Years)")

fig.legend(loc="upper right")
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(1)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_ra",
    scatter=False,
    ax=ax,
    label="RA",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_ract1",
    scatter=False,
    ax=ax,
    label="RACT1",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_ract1r3",
    scatter=False,
    ax=ax,
    label="RACT1-R3",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_ract1r5",
    scatter=False,
    ax=ax,
    label="RACT1-R5",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_ract2",
    scatter=False,
    ax=ax,
    label="RACT2",
)


fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_ra",
    scatter=False,
    ax=ax,
    label="RA",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_ract1",
    scatter=False,
    ax=ax,
    label="RACT1",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_ract1r3",
    scatter=False,
    ax=ax,
    label="RACT1-R3",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_ract1r5",
    scatter=False,
    ax=ax,
    label="RACT1-R5",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_ract2",
    scatter=False,
    ax=ax,
    label="RACT2",
)


fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(
    global_df.age_years,
    global_df["Subcortical gray matter volume_ract1"],
    ".",
    label="Subcort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_ract1",
    scatter=False,
    ax=ax,
    color=cp[0],
)
ax.plot(
    global_df.age_years,
    global_df["Total cortical gray matter volume_ract1"],
    ".",
    label="Cort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_ract1",
    scatter=False,
    ax=ax,
    color=cp[1],
)

ax.plot(
    global_df.age_years,
    global_df["Total cerebral white matter volume_ract1"],
    ".",
    label="Cereb WMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cerebral white matter volume_ract1",
    scatter=False,
    ax=ax,
    color=cp[2],
)

ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of recon-all clinical from recon-all")
fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(
    global_df.age_years,
    global_df["Subcortical gray matter volume_ract1r5"],
    ".",
    label="Subcort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_ract1r5",
    scatter=False,
    ax=ax,
    color=cp[0],
)
ax.plot(
    global_df.age_years,
    global_df["Total cortical gray matter volume_ract1r5"],
    ".",
    label="Cort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_ract1r5",
    scatter=False,
    ax=ax,
    color=cp[1],
)

ax.plot(
    global_df.age_years,
    global_df["Total cerebral white matter volume_ract1r5"],
    ".",
    label="Cereb WMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cerebral white matter volume_ract1r5",
    scatter=False,
    ax=ax,
    color=cp[2],
)

ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of recon-all clinical from recon-all")
fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(
    global_df.age_years,
    global_df["Subcortical gray matter volume_pct_ra_dif_ct1"],
    ".",
    label="Subcort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Subcortical gray matter volume_pct_ra_dif_ct1",
    scatter=False,
    ax=ax,
    color=cp[0],
)
ax.plot(
    global_df.age_years,
    global_df["Total cortical gray matter volume_pct_ra_dif_ct1"],
    ".",
    label="Cort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cortical gray matter volume_pct_ra_dif_ct1",
    scatter=False,
    ax=ax,
    color=cp[1],
)

ax.plot(
    global_df.age_years,
    global_df["Total cerebral white matter volume_pct_ra_dif_ct1"],
    ".",
    label="Cereb WMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y="Total cerebral white matter volume_pct_ra_dif_ct1",
    scatter=False,
    ax=ax,
    color=cp[2],
)

ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of recon-all clinical from recon-all")
ax.set_ylim((-20, 20))
fig.legend()

## Multi panels

In [ ]:
# T2
source = "not2"
fig, axes = plt.subplots(2, 2, figsize=(10, 10), dpi=100)
ax = axes[0, 0]
metric = "Total cortical gray matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
# g.set_xlim(xlim)
# g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all noT2")
g.set_title(f"Cort GMV, $r^2={r2:0.2f}$")

ax = axes[0, 1]
metric = "Total cerebral white matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
# g.set_xlim(xlim)
# g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all noT2")
g.set_title(f"Cereb WMV $r^2={r2:0.2f}$")

ax = axes[1, 0]
metric = "Subcortical gray matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
# g.set_xlim(xlim)
# g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all noT2")
g.set_title(f"Subcort GMV, $r^2={r2:0.2f}$")

ax = axes[1, 1]

ax.plot(
    global_df.age_years,
    global_df[f"Total cortical gray matter volume_pct_ra_dif_{source}"],
    ".",
    label="Cort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Total cortical gray matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[0],
)

ax.plot(
    global_df.age_years,
    global_df[f"Total cerebral white matter volume_pct_ra_dif_{source}"],
    ".",
    label="Cereb WMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Total cerebral white matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[1],
)
ax.plot(
    global_df.age_years,
    global_df[f"Subcortical gray matter volume_pct_ra_dif_{source}"],
    ".",
    label="Subcort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Subcortical gray matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[2],
)

ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of recon-all noT2 from recon-all")
# ax.set_ylim((-20, 20))
ax.legend()
fig.tight_layout()

In [ ]:
source = "ct1r3"
fig, axes = plt.subplots(2, 2, figsize=(10, 10), dpi=100)
ax = axes[0, 0]
metric = "Total cortical gray matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Cort GMV, $r^2={r2:0.2f}$")

ax = axes[0, 1]
metric = "Total cerebral white matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
# g.set_xlim(xlim)
# g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Cereb WMV $r^2={r2:0.2f}$")

ax = axes[1, 0]
metric = "Subcortical gray matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
# g.set_xlim(xlim)
# g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Subcort GMV, $r^2={r2:0.2f}$")

ax = axes[1, 1]

ax.plot(
    global_df.age_years,
    global_df[f"Total cortical gray matter volume_pct_ra_dif_{source}"],
    ".",
    label="Cort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Total cortical gray matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[0],
)

ax.plot(
    global_df.age_years,
    global_df[f"Total cerebral white matter volume_pct_ra_dif_{source}"],
    ".",
    label="Cereb WMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Total cerebral white matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[1],
)
ax.plot(
    global_df.age_years,
    global_df[f"Subcortical gray matter volume_pct_ra_dif_{source}"],
    ".",
    label="Subcort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Subcortical gray matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[2],
)

ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of recon-all clinical from recon-all")
# ax.set_ylim((-20, 20))
ax.legend()
fig.tight_layout()

In [ ]:
source = "ct1r5"
fig, axes = plt.subplots(2, 2, figsize=(10, 10), dpi=100)
ax = axes[0, 0]
metric = "Total cortical gray matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Cort GMV, $r^2={r2:0.2f}$")

ax = axes[0, 1]
metric = "Total cerebral white matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Cereb WMV $r^2={r2:0.2f}$")

ax = axes[1, 0]
metric = "Subcortical gray matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Subcort GMV, $r^2={r2:0.2f}$")

ax = axes[1, 1]

ax.plot(
    global_df.age_years,
    global_df[f"Total cortical gray matter volume_pct_ra_dif_{source}"],
    ".",
    label="Cort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Total cortical gray matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[0],
)

ax.plot(
    global_df.age_years,
    global_df[f"Total cerebral white matter volume_pct_ra_dif_{source}"],
    ".",
    label="Cereb WMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Total cerebral white matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[1],
)
ax.plot(
    global_df.age_years,
    global_df[f"Subcortical gray matter volume_pct_ra_dif_{source}"],
    ".",
    label="Subcort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Subcortical gray matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[2],
)

ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of recon-all clinical from recon-all")
ax.set_ylim((-20, 20))
ax.legend()
fig.tight_layout()

In [ ]:
source = "ct1"
fig, axes = plt.subplots(2, 2, figsize=(10, 10), dpi=100)
ax = axes[0, 0]
metric = "Total cortical gray matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Cort GMV, $r^2={r2:0.2f}$")

ax = axes[0, 1]
metric = "Total cerebral white matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Cereb WMV $r^2={r2:0.2f}$")

ax = axes[1, 0]
metric = "Subcortical gray matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Subcort GMV, $r^2={r2:0.2f}$")

ax = axes[1, 1]

ax.plot(
    global_df.age_years,
    global_df[f"Total cortical gray matter volume_pct_ra_dif_{source}"],
    ".",
    label=r"Cort GMV, $\beta=0.40, t=15.78, p=3.15e^{-50}$",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Total cortical gray matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[0],
)

ax.plot(
    global_df.age_years,
    global_df[f"Subcortical gray matter volume_pct_ra_dif_{source}"],
    ".",
    label=r"Subcort GMV, $\beta=0.014, t=0.36, p=0.72$",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Subcortical gray matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[1],
)
ax.plot(
    global_df.age_years,
    global_df[f"Total cerebral white matter volume_pct_ra_dif_{source}"],
    ".",
    label=r"Cereb WMV, $\beta=-0.17, t=-6.24, p=6.43e^{-10}$",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Total cerebral white matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[2],
)


ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of recon-all clinical from recon-all")
ax.set_ylim((-20, 20))
ax.legend()
fig.suptitle("recon-all vs recon-all clinical with high-res T1w input")
fig.tight_layout()

In [ ]:
source = "ct1r5"
fig, axes = plt.subplots(2, 2, figsize=(10, 10), dpi=100)
ax = axes[0, 0]
metric = "Total cortical gray matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Cort GMV, $r^2={r2:0.2f}$")

ax = axes[0, 1]
metric = "Total cerebral white matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Cereb WMV $r^2={r2:0.2f}$")

ax = axes[1, 0]
metric = "Subcortical gray matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Subcort GMV, $r^2={r2:0.2f}$")

ax = axes[1, 1]

ax.plot(
    global_df.age_years,
    global_df[f"Total cortical gray matter volume_pct_ra_dif_{source}"],
    ".",
    label="Cort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Total cortical gray matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[0],
)

ax.plot(
    global_df.age_years,
    global_df[f"Total cerebral white matter volume_pct_ra_dif_{source}"],
    ".",
    label="Cereb WMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Total cerebral white matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[1],
)
ax.plot(
    global_df.age_years,
    global_df[f"Subcortical gray matter volume_pct_ra_dif_{source}"],
    ".",
    label="Subcort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Subcortical gray matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[2],
)

ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of recon-all clinical from recon-all")
ax.set_ylim((-20, 20))
ax.legend()
fig.tight_layout()

In [ ]:
source = "ct2"
fig, axes = plt.subplots(2, 2, figsize=(10, 10), dpi=100)
ax = axes[0, 0]
metric = "Total cortical gray matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Cort GMV, $r^2={r2:0.2f}$")

ax = axes[0, 1]
metric = "Total cerebral white matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Cereb WMV $r^2={r2:0.2f}$")

ax = axes[1, 0]
metric = "Subcortical gray matter volume"
x = f"{metric}_ra"
y = f"{metric}_ra{source}"
r2 = (
    np.corrcoef(
        global_df.loc[global_df[y].notnull(), x],
        global_df.loc[global_df[y].notnull(), y],
    )[0, 1]
    ** 2
)
g = sns.scatterplot(data=global_df, x=x, y=y, hue="age_years", marker="o", ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all")
g.set_ylabel("recon-all clinical")
g.set_title(f"Subcort GMV, $r^2={r2:0.2f}$")

ax = axes[1, 1]

ax.plot(
    global_df.age_years,
    global_df[f"Total cortical gray matter volume_pct_ra_dif_{source}"],
    ".",
    label="Cort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Total cortical gray matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[0],
)

ax.plot(
    global_df.age_years,
    global_df[f"Total cerebral white matter volume_pct_ra_dif_{source}"],
    ".",
    label="Cereb WMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Total cerebral white matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[1],
)
ax.plot(
    global_df.age_years,
    global_df[f"Subcortical gray matter volume_pct_ra_dif_{source}"],
    ".",
    label="Subcort GMV",
)
sns.regplot(
    data=global_df,
    x="age_years",
    y=f"Subcortical gray matter volume_pct_ra_dif_{source}",
    scatter=False,
    ax=ax,
    color=cp[2],
)

ax.set_xlabel("Age (years)")
ax.set_ylabel("% Difference of recon-all clinical from recon-all")
ax.set_ylim((-20, 20))
ax.legend()
fig.tight_layout()

## Single panel correlation

In [ ]:
metric = "Total cortical gray matter volume"
g = sns.scatterplot(
    data=global_df.loc[global_df[f"{metric}_ract1r5"].notnull()],
    x=f"{metric}_ract1",
    y=f"{metric}_ract1r5",
    hue="age_years",
)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all from highres T1")
g.set_ylabel("recon-all clinical from highres T2")
g.set_title(f"recon-all clinical estimated\n{metric}")

In [ ]:
metric = "Total cortical gray matter volume"
g = sns.scatterplot(
    data=global_df, x=f"{metric}_ra", y=f"{metric}_ract2", hue="age_years"
)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("recon-all from highres T1")
g.set_ylabel("recon-all clinical from highres T2")
g.set_title(f"recon-all clinical estimated\n{metric}")

In [ ]:
metric = "Total cerebral white matter volume"
g = sns.scatterplot(
    data=global_df, x=f"{metric}_ract1", y=f"{metric}_ract2", hue="age_years"
)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel("Estimated from highres T1")
g.set_ylabel("Estimated from highres T2")
g.set_title(f"recon-all clinical estimated\n{metric}")

In [ ]:
metric = "Subcortical gray matter volume"
g = sns.scatterplot(
    data=global_df, x=f"{metric}_ract1", y=f"{metric}_ract2", hue="age_years"
)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_ylabel("Estimated from highres T1")
g.set_xlabel("Estimated from highres T2")
g.set_title(f"recon-all clinical estimated\n{metric}")

In [ ]:
metric = "Total gray matter volume"
g = sns.scatterplot(
    data=global_df, x=f"{metric}_ract1", y=f"{metric}_ract2", hue="age_years"
)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 200], [0, 200], label="x=y", color="black")
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_ylabel("Estimated from highres T1")
g.set_xlabel("Estimated from highres T2")
g.set_title(f"recon-all clinical estimated\n{metric}")

## Age vn

In [ ]:
global_resses.query("vn == 'age_years' & dif == 't1'").sort_values(["metric", "dif"])

In [ ]:
global_resses.query("vn == 'age_years' & dif == 't2'").sort_values(["metric", "dif"])

In [ ]:
dir(g.legend_.legend_handles)